Transformer RAG 파이프라인 간단 요약
- Retrieval 단계 : Qdrant 의미 기반 검색엔진으로 관련 문서를 빠르게 찾아냄
- Augmentation 단계 : 검색된 문서를 LLM 입력 프롬프트에 추가해 맥락 강화
- Generation 단계 : LLM이 최종 답변을 생성 → 정확성과 신뢰성 향상

RAG 파이프라인
1. 데이터 수집 및 저장
	- PostgreSQL 테이블 생성 및 입력
	- DB 조회 + 로깅 설정으로 데이터 관리
2. Qdrant 의미 기반 검색 구축
	- 뉴스 컬렉션 생성
	- Qdrant 서버 실행 및 API 테스트
3. 임베딩 생성 및 삽입
	- SentenceTransformer 임베딩 생성
	- Batch 단위 Insert/Update
4. 검색(Retrieval)
    - 질의 임베딩 생성 후 top-k 문서 검색
5. QA 처리
	- KoELECTRA QA 모델 + MeCab 형태소 분석
	- 입력: input_ids, attention_mask
	- 출력: 자연어 응답 복원
6. 요약 처리
	- KoBART Summarization 모델
	- 반복 억제, 길이 조절, 다양성 확보
	- 후처리: clean_summary
7. 최종 응답 
    - QA + 요약 + 출처 정보 포함
    - 외부 서비스와 연계해 자연스러운 문장으로 재구성 필요
8. 서비스 구성
	- FastAPI 실행 및 /search 엔드포인트 제공
	- 출력: QA + 요약 결과 + 출처 정보

파이프라인 흐름도
```mermaid
flowchart TD
    A[사용자 질의] --> B[FastAPI 엔드포인트 /search]
    B --> C[SentenceTransformer 임베딩 생성]
    C --> D[Qdrant 의미 기반 검색]
    D --> E[관련 문서 반환]
    E --> F[KoELECTRA QA 모델 + MeCab 형태소 분석]
    F --> G[KoBART Summarization + clean_summary]
    G --> H[응답 + 출처 표시]
    H --> I[외부 LLM 서비스 연계 검토]
    I --> J[최종 사용자 응답]
```


In [1]:
# 데이터 수집 모듈 구현
import feedparser               # RSS 피드를 파싱해서 뉴스 항목을 가져옴
from bs4 import BeautifulSoup   # HTML 태그 제거 및 텍스트 정제
from datetime import datetime
import psycopg2                 # PostgreSQL DB 연결 및 SQL 실행
import logging                  # 실행 과정 기록 (파일 + 콘솔 출력)

# RSS 가져오기 함수
def fetch_rss(rss_url: str):
    # feedparser 파싱 -> 뉴스 항목(entries) 리스트 반환
    feed = feedparser.parse(rss_url)
    return feed.entries

# 뉴스 파싱 함수
def parse_entry(entry):
    title = entry.title # 뉴스 제목, RSS 항목의 <title> 태그에 해당
    content_raw = entry.description if "description" in entry else ""  # 뉴스 본문, RSS 항목에 description 속성, 없으면 빈 문자열 반환
    content = BeautifulSoup(content_raw, "html.parser").get_text() # 텍스트만 추출, HTML 제거
    url = entry.link # 뉴스 원문 링크(URL), RSS 항목의 <link> 태그
    published_at = datetime(*entry.published_parsed[:6]) if hasattr(entry, "published_parsed") else datetime.now() # entry.published_parsed가 있으면 datetime 객체로 변환(연, 월, 일, 시, 분, 초)
    source_name = entry.source.title if hasattr(entry, "source") else "Google News" # entry.source가 있으면 그 안의 title을 사용하고, 없으면 기본값 "Google News"로 설정
    source_url = entry.source.href if hasattr(entry, "source") else "" # entry.source가 있으면 href 속성을 사용하고, 없으면 빈 문자열로 처리

    return title, content, url, published_at, source_name, source_url

# DB Insert 함수
def insert_article(cur, conn, title, content, url, published_at, source_name, source_url):
    try: # SQL 실행
        cur.execute("""
            INSERT INTO news_articles (title, content, url, published_at, source_name, source_url)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT (url) DO NOTHING;
        """, (title, content, url, published_at, source_name, source_url))
        if cur.rowcount > 0: # cur.rowcount → SQL 실행 후 영향을 받은 행 수
            logging.info("데이터 Insert 성공: %s", title) # 1 이상이면 새로운 데이터가 추가된 것 → “Insert 성공” 로그 기록
        else:
            logging.info("중복으로 건너뜀: %s", title) # 0이면 중복으로 인해 추가되지 않은 것 → “중복으로 건너뜀” 로그 기록
        conn.commit() # 실행 결과를 실제 DB 반영
    except Exception as e:
        logging.error("Insert 실패: %s, 에러: %s", title, e)

# 로깅 설정: 파일 + 콘솔 출력
logging.basicConfig(
    level=logging.INFO, # INFO 이상 레벨 기록
    format="%(asctime)s [%(levelname)s] %(message)s", # 출력 형식
    handlers=[
        logging.FileHandler("24.transformer_rag3_app.log", encoding="utf-8"), # 파일 저장
        logging.StreamHandler() # 콘솔 출력
    ]
)
    
# RSS 함수 호출
rss_url = "https://news.google.com/rss?hl=ko&gl=KR&ceid=KR:ko"
entries = fetch_rss(rss_url)

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 뉴스 파싱 함수 -> DB Insert 함수
for entry in entries:
    title, content, url, published_at, source_name, source_url = parse_entry(entry)
    insert_article(cur, conn, title, content, url, published_at, source_name, source_url)

cur.close()
conn.close()
logging.info("DB 연결 종료 완료")

2026-05-19 10:49:27,762 [INFO] 데이터 Insert 성공: “정용진 ‘극우’ 행보 탓에 비난 큰 건데…대표 경질은 물타기” - 한겨레
2026-05-19 10:49:27,775 [INFO] 중복으로 건너뜀: 접점 못 찾는 미-이란…‘재촉’ 트럼프 “새 협상안 빨리 안 내놓으면 공격” - 한겨레
2026-05-19 10:49:27,783 [INFO] 데이터 Insert 성공: 이 대통령, 안동 찾는 다카이치 총리에 ‘하회탈·한지가죽가방’ 선물 - 경향신문
2026-05-19 10:49:27,790 [INFO] 데이터 Insert 성공: 영남 중심으로 덥고 전국 ‘흐림’… 내일은 비 - 한겨레
2026-05-19 10:49:27,800 [INFO] 데이터 Insert 성공: 이 대통령 "5·18 정신, 헌법에 당당히 새겨야"…전남도청 복원 개관 - 대한민국 정책브리핑
2026-05-19 10:49:27,807 [INFO] 데이터 Insert 성공: 민주콩고·우간다 에볼라 사망 100명 넘어…아프리카CDC도 비상사태 선언 - 한겨레
2026-05-19 10:49:27,819 [INFO] 데이터 Insert 성공: '철근 누락'에 GTX-A 8월 개통 불투명…경기 직장인·집주인 한숨 - 뉴스1
2026-05-19 10:49:27,828 [INFO] 데이터 Insert 성공: “닿지 못한 목소리에도 귀를”…‘이재명 정부 1년’ 대통령에 바란다 - 경향신문
2026-05-19 10:49:27,838 [INFO] 데이터 Insert 성공: 정원오 40% 오세훈 37%, 전재수 44% 박형준 35% - 조선일보
2026-05-19 10:49:27,844 [INFO] 데이터 Insert 성공: [단독]게임 제작진이 캐릭터 속옷 비추며 “군침 돈다” 방송…아동·청소년 단체 “아동 대상 성애화 만연” - 경향신문
2026-05-19 10:49:27,854 [INFO] 데이터 Insert 성공: "중국 소유 선박도 공격받아"…드론·미사일 쏟아부은 러시아 - 

In [2]:
# 뉴스 조회
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(dbname="newsdb", user="newsuser", password="1234", host="localhost", port="5432")
cur = conn.cursor()

# 1. 전체 뉴스 개수 조회
try:
    cur.execute("""
            SELECT COUNT(*) 
            FROM news_articles;
    """)
    count = cur.fetchone()[0]
    logging.info("총 뉴스 개수: %s", count)
except Exception as e:
    logging.error("총 뉴스 개수 중 에러 발생:", e)

# 2. 최신 뉴스 5개 조회
try:
    cur.execute("""
            SELECT title, published_at
            FROM news_articles
            ORDER BY published_at DESC
            LIMIT 5;
    """)
    lastest_news = cur.fetchall()
    logging.info("최신 뉴스 5개:")
    for row in lastest_news:
        logging.info("제목: %s | 발행일: %s", row[0], row[1])
except Exception as e:
    logging.error("최신 뉴스 검색 중 에러 발생:", e)

# 3. 특정 키워드 검색(예시: AI)
try:
    keyword = 'AI'
    cur.execute("""
                SELECT title, url
                FROM news_articles
                WHERE content LIKE %s
                ORDER BY published_at DESC
                LIMIT 10;
    """, (f"%{keyword}%",)) # 튜플 형식 지켜야 에러가 안남
    keyword_news = cur.fetchall()
    logging.info("키워드 '%s' 관련 뉴스:", keyword)
    for row in keyword_news:
        logging.info("제목: %s | URL: %s", row[0], row[1])
except Exception as e:
    logging.error("키워드 검새 중 에러 발생:", e)

# 4. 출처별 뉴스 개수
try:
    cur.execute(""" 
            SELECT source_name, COUNT(*)
            FROM news_articles
            GROUP BY source_name
            ORDER BY COUNT(*) DESC;
    """)
    source_stats = cur.fetchall()
    logging.info("출처별 뉴스 개수:")
    for row in source_stats:
        logging.info("출처: %s | 개수: %s", row[0], row[1])
except Exception as e:
    logging.error("출처별 뉴스 개수 검색 중 에러 발생:", e)

cur.close()
conn.close()
logging.info("조회 완료 및 DB 연결 종료")

2026-05-19 10:49:28,158 [INFO] 총 뉴스 개수: 514
2026-05-19 10:49:28,169 [INFO] 최신 뉴스 5개:
2026-05-19 10:49:28,170 [INFO] 제목: 이 대통령, 안동 찾는 다카이치 총리에 ‘하회탈·한지가죽가방’ 선물 - 경향신문 | 발행일: 2026-05-19 01:07:00
2026-05-19 10:49:28,173 [INFO] 제목: 北내고향여자축구단, 내일 ‘수원FC위민’과 남북 맞대결 - BBS불교방송 | 발행일: 2026-05-19 01:03:02
2026-05-19 10:49:28,175 [INFO] 제목: 고대의대, 질병청 ‘한타바이러스 백신 개발 과제’ 주관기관 선정 - 메드월드뉴스 | 발행일: 2026-05-19 00:50:56
2026-05-19 10:49:28,177 [INFO] 제목: 에이수스, 2026년형 ROG Strix Scar·Strix G 시리즈 국내 선보여 - elec4 | 발행일: 2026-05-19 00:46:33
2026-05-19 10:49:28,180 [INFO] 제목: "급성췌장염, 재발 막아야 만성 진행 막는다" - 보건신문 | 발행일: 2026-05-19 00:44:07
2026-05-19 10:49:28,188 [INFO] 키워드 'AI' 관련 뉴스:
2026-05-19 10:49:28,189 [INFO] 제목: 비수로 끝난 ‘AI 브로맨스’...머스크 vs 올트먼 법정 싸움 승자는 - 조선일보 | URL: https://news.google.com/rss/articles/CBMigwFBVV95cUxOaUpYTU9hY29uNW1KSTRZMnZRZzRNRDVpU05oallEeGgwcTRKN1RlNHRUVERlTnBmbExwVWt0OVFnTjU0Tjh5Qm9SQUxnT2FYMldRMG1UbzB0YUgzTjBJenl4MEdXUUJPeHJrS3FjTmR5d2JkM0lwZU5hemlIeEtxMEZENA?oc=5
2026-05-19 10:49:28,190 [

In [3]:
#  색인용 데이터 추출
import psycopg2
import logging

# DB 연결
conn = psycopg2.connect(
    dbname="newsdb", 
    user="newsuser", 
    password="1234", 
    host="localhost", 
    port="5432"
)

# cursor 객체 생성
cur = conn.cursor()

try:
    # 색인용 데이터 조회, 최근 데이터 100개 기준
    cur.execute(""" 
                SELECT id, title, content, url, published_at, source_name
                FROM news_articles
                ORDER BY published_at DESC
                LIMIT 100;
    """)
    articles = cur.fetchall()
    logging.info("색인용 데이터 %s개 가져옴:", len(articles))

    # Qdrant에 넣을 준비: 리스트 형태로 변환
    index_data = []
    for row in articles:
        record = {
            "id": row[0],
            "title": row[1],
            "content": row[2],
            "url": row[3],
            "published_at": row[4],
            "source_name": row[5]
        }
        index_data.append(record)

except Exception as e:
    logging.error("색인용 조회 에러 발생: %s", e)

# DB connect 종료
cur.close()
conn.close()
logging.info("색인 데이터 조회 완료 및 DB 연결 종료")

2026-05-19 10:49:28,918 [INFO] 색인용 데이터 100개 가져옴:
2026-05-19 10:49:28,921 [INFO] 색인 데이터 조회 완료 및 DB 연결 종료


In [4]:
# Qdrant Collection 생성
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
import logging

# Qdrant 연결
qdrant = QdrantClient(host="localhost", port=6333)

# Qdrant Collection 초기화 + 생성
# qdrant.recreate_collection( # 지정한 이름의 컬렉션을 새로 생성, 이미 같은 이름의 컬렉션이 있으면 삭제 후 다시 생성(초기화 + 생성 기능)
#     collection_name="news_articles", # 컬렉션 명
#     vectors_config=VectorParams( # 벡터 설정 정의
#         size=384, # 모델 MiniLM-L12-v2 임베딩 차원 수(384차원)
#         distance=Distance.COSINE # 벡터 유사도 계산 방식(COSINE 추천, 의미 유사도 검색에 가장 많이 사용), DOT(내적 기반), EUCLID(거기 기반)
#     )
# )

# Qdrant Collecton 생성
if not qdrant.collection_exists("news_articles"):
    qdrant.create_collection(
        collection_name="news_articles", # 컬렉션 명
        vectors_config=VectorParams( # 벡터 설정 정의
            size=384, # 모델 MiniLM-L12-v2 임베딩 차원 수(384차원)
            distance=Distance.COSINE # 벡터 유사도 계산 방식(COSINE 추천, 의미 유사도 검색에 가장 많이 사용), DOT(내적 기반), EUCLID(거기 기반)
        )
    )
    logging.info(f"Qdrant Collection 'news_articles' 생성 완료")
else:
    logging.info(f"Qdrant Collection 'news_articles' 존재 합니다.")

/Users/ai/deep-learning/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
2026-05-19 10:49:31,896 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-05-19 10:49:31,929 [INFO] HTTP Request: GET http://localhost:6333/collections/news_articles/exists "HTTP/1.1 200 OK"
2026-05-19 10:49:31,935 [INFO] Qdrant Collection 'news_articles' 존재 합니다.


In [5]:
# Qdrant news_articles 컬렉션 update/insert
import torch
from sentence_transformers import SentenceTransformer # Hugging Face sentence-transformer 라이브러리 사용
from qdrant_client import QdrantClient
from qdrant_client.http import models # qdrant 컬렉션 및 검색 관련 설정 정의하는 데이터 모델(구조체) 로드
import logging

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logging.info(f"Pytorch Version : {torch.__version__}, Device : {device}")

# 다국어 임베딩 모델 로드
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)

# 임베딩 함수 생성
def generate_embedding(text: str):
    # 본문(content)을 벡터로 변환
    embedding = model.encode(text)
    return embedding.tolist() # Qdrant에 넣기 위해 list 변환

# 데이터 저장 형식 설명
# qdrant.upsert()는 삽입(insert)+업데이트(update) 기능을 동시에 수행, 같은 ID가 있으면 덮어쓰고, 없으면 새로 추가한다
    # { 
    #     "id": 1,
    #     "vector": [0.123, -0.456, 0.789, ...],
    #     "payload": {
    #         "title": "AI 의료 활용",
    #         "content": "AI가 의료 분야에서 활용되는 사례...",
    #         "date": "2026-03-11",
    #         "author": "홍길동"
    #     }
    # }
# Qdrant update/insert 함수
def insert_to_qdrant(data):
    ids = []
    vectors = []
    payloads = []
    
    for item in data:
        # 본문만 임베딩
        text = item["content"]
        embedding = generate_embedding(text=text) # 임베딩 함수 호출

        ids.append(item["id"]) # ID 값
        vectors.append(embedding) # 임베딩 처리 후 벡터 값
        payloads.append(item) # 원본 데이터
    
    qdrant.upsert(
        collection_name="news_articles",
        points=models.Batch(
            ids=ids,
            vectors=vectors,
            payloads=payloads # paylods(JSON) 형태로 저장
        )
    )
    logging.info("본문 임베딩 및 Qdrant 저장 완료")
    

# Batch 단위로 Qdrant update/insert
batch_size = 30
total = len(index_data)

for i in range(0, len(index_data), batch_size):
    chunk = index_data[i : i + batch_size] # 2개씩 잘라서 가져온다
    insert_to_qdrant(chunk) # 잘라낸 청크를 Qdrant에 전달

    # 진행 상황 계산
    start_id = i
    end_id = i + len(chunk) - 1
    processed = end_id + 1
    progress = (processed / total) * 100
    remaining = total - processed

    logging.info(
        f"Batch {start_id} ~ {end_id} 인덱싱 완료" 
        f"(총 {len(chunk)}건), 진행률 : {progress:.2f}%, 남은 데이터 : {remaining}건"
    )

# 최종 데이터 개수 확인
count_result = qdrant.count(collection_name="news_articles")
logging.info(f"Qdrant news_articles 컬렉션 최종 저장 건수 : {count_result}건")

2026-05-19 10:49:43,268 [INFO] Pytorch Version : 2.2.2, Device : cpu
2026-05-19 10:49:43,322 [INFO] Use pytorch device_name: cpu
2026-05-19 10:49:43,323 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
2026-05-19 10:49:48,582 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-19 10:50:02,542 [INFO] HTTP Request: PUT http://localhost:6333/collections/news_articles/points?wait=true "HTTP/1.1 200 OK"
2026-05-19 10:50:02,546 [INFO] 본문 임베딩 및 Qdrant 저장 완료
2026-05-19 10:50:02,547 [INFO] Batch 90 ~ 99 인덱싱 완료(총 10건), 진행률 : 100.00%, 남은 데이터 : 0건
2026-05-19 10:50:02,571 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/count "HTTP/1.1 200 OK"
2026-05-19 10:50:02,586 [INFO] Qdrant news_articles 컬렉션 최종 저장 건수 : count=333건


In [6]:
# Qdrant 검색(Query)

# Qdrant 검색 함수
def search_qdrant(query: str, top_k: int = 5):
    # 질의 임베딩 생성
    query_vector = generate_embedding(query)

    # qdrant에서 검색
    result = qdrant.query_points(
        collection_name="news_articles",
        query=query_vector,
        limit=top_k
    )
    # result → QueryResponse 객체, 검색 결과를 감싸는 객체
    # result.points → ScoredPoint 리스트 : 실제 결과는 result.points에 들어 있는 ScoredPoint 리스트, score, .payload 등을 꺼내 쓰면 된다
    return result.points # ScoredPoint 리스트 반환

# qdrant 검색 함수 호출
search_result = search_qdrant("이란 전쟁 상황 알려줘")

# 결과 출력
for r in search_result:
    logging.info(
        f"Score : {r.score:.4f}, Title : {r.payload.get('title')}, "
        f"Source : {r.payload.get('source_name')} , URL : {r.payload.get('url')} "
    )

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-19 10:50:02,814 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/query "HTTP/1.1 200 OK"
2026-05-19 10:50:02,820 [INFO] Score : 0.7748, Title : 이란 “美 재공격시 신속·강력 대응... 하메네이에 ‘새 지침’ 수령” - 조선일보, Source : 조선일보 , URL : https://news.google.com/rss/articles/CBMingFBVV95cUxPdldabVNNaXNOMlQtUVQ3dERkcHRUV3RpV1dFcUJTUGNSQy00RWRxeTkyZEgtT1VvWkpUeG1Yd1lraWlqVjgwMmxUWExDbk9fa2s5U2FLOEpqOU04bXBVenB2b2U1dWFZbGVacUhMV1IzN0hLS0lvRlQ5TjRXZHFya190N0wzWDAtRnJreDhhZTVyaUtPTVV5QWwySzhYUQ?oc=5 
2026-05-19 10:50:02,822 [INFO] Score : 0.7024, Title : 코스피, 장 초반 7800 돌파…반도체 강세, 매수 사이드카 발동 - 한겨레, Source : 한겨레 , URL : https://news.google.com/rss/articles/CBMiZ0FVX3lxTE1QNHFtTW1IdTFpbWw4QU5CZmd0WXBJN05ZdlA4a29rZmxYRUJYOWZ0aFg2alpabklyWENGRFh0aDFzaHVWU3MxS09mMUlYTFZ4VTNWQkt6cW4yby1HblA1dkJVWi00aHM?oc=5 
2026-05-19 10:50:02,823 [INFO] Score : 0.6871, Title : 이란, 트럼프 합의 압박 아랑곳… 되레 "전쟁 배상금 달라" - 조선일보, Source : 조선일보 , URL : https://news.google.com/rss/articles/CBMingFBVV95cUxPT3Ytc

In [7]:
# 형태소 분석 테스트

# MeCab 설치 (macOS 기준) : brew install mecab mecab-ipadic
# 파이썬 바인딩 설치 : pip install mecab-python3

# 형태소 분석 테스트
from mecab import MeCab

mecab = MeCab()
print(mecab.morphs("인공지능은 의료 분야에서 활용된다."))
print(mecab.nouns("인공지능은 의료 분야에서 활용된다."))
print(mecab.pos("인공지능은 의료 분야에서 활용된다."))

['인공지능', '은', '의료', '분야', '에서', '활용', '된다', '.']
['인공지능', '의료', '분야', '활용']
[('인공지능', 'NNP'), ('은', 'JX'), ('의료', 'NNG'), ('분야', 'NNG'), ('에서', 'JKB'), ('활용', 'NNG'), ('된다', 'XSV+EF'), ('.', 'SF')]


In [ ]:
# QA 모델 적용 : embedding + qdrant + QA Model
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from mecab import MeCab
import gc
import re

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logging.info(f"PyTorch Version : {torch.__version__}, Device : {device}")

# embedding
embedder_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder_model = SentenceTransformer(embedder_name)

# KorQuAD (Korean Question Answering Dataset) : 한국어 위키 문서를 기반으로 만든 QA 데이터셋으로 질문과 답변이 문맥 내에서 매핑되어 있다
qa_model_name = "monologg/koelectra-base-v3-finetuned-korquad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name) # QA Tokenizer
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name) # QA Model

# embedding 함수
def generate_embedding(text: str):
    embedding = embedder_model.encode(text) # 임베딩 벡터 변환
    return embedding.tolist()

# qdrant 검색 함수
def search_qdrant(query: str, top_k: int = 5):
    # 질의 임베딩 생성
    query_vector = generate_embedding(query)

    # qdrant 검색
    query_result = qdrant.query_points(
        collection_name="news_articles",
        query=query_vector,
        limit=top_k
    )
    # query_result : QueryResponse 객체, 검색 결과를 감싸는 객체
    # query_result.points : ScoredPoint 리스트 실제 결과는 query_result.points에 들어 있는 ScoredPoint 리스트, score, .payload 등을 꺼내 쓰면 된다
    return query_result.points

# QA 모델 파이프라인: 
# QA 토크나이저 > QA 모델 > 답변 시작/끝 위치 > 토큰 ID를 토큰 문자열 변환 > 특수 토큰 제거 > 토큰을 문자열로 변환
def qa_pipeline(question, context, top_k=3, max_answer_len=80):
    # 토크나이징: 질문과 문서를 토큰 ID로 변환
    inputs = qa_tokenizer(
        question, # 질문
        context, # 문서
        return_tensors="pt", # 파이토치 형태로 리턴
        max_length=512, # 입력 최대 길이
        truncation=True, # 길면 잘라내기
        padding="max_length"
    ).to(device)

    # QA 모델 추론
    with torch.no_grad():
        # QA 모델 추론
        outputs = qa_model(**inputs) # **inputs 파라미터 형태로 입력
    
    # # 후처리 1. 답변 시작/끝 위치 예측하는 확률 분포
    # answer_start = torch.argmax(outputs.start_logits)
    # answer_end = torch.argmax(outputs.end_logits) + 1
    # # logging.info(f"answer_start : {answer_start}")
    # # logging.info(f"answer_end : {answer_end}")

    # # 후처리 2. 시작/끝 위치를 잡을때, 최소/최대 길이를 강제로 설정(최소 5토큰 이상, 최대 50토큰 이하)
    # if answer_end - answer_start < 5:
    #     answer_end = answer_start + 5
    # elif answer_end - answer_start > 50:
    #     answer_end = answer_start + 50
    # # logging.info(f"answer_start 조건 추가 : {answer_start}")
    # # logging.info(f"answer_end 조건 추가 : {answer_end}")

    # # 후처리 3. 토큰ID > 토큰 문자열 변환
    # tokens = qa_tokenizer.convert_ids_to_tokens(
    #     inputs["input_ids"][0][answer_start:answer_end]
    # )
    # # logging.info(f"tokens 토큰ID > 토큰 문자열 변환 : {tokens}")

    # # 후처리 4. [CLS],[SEP],[PAD] 특수 토큰 제거
    # tokens = [t for t in tokens if t not in qa_tokenizer.all_special_tokens]
    # # logging.info(f"tokens [CLS],[SEP],[PAD] 특수 토큰 제거 : {tokens}")

    # # 후처리 5. 토큰 > 최종적으로 사람이 읽을 수 있는 문자열로 복원
    # answer = qa_tokenizer.convert_tokens_to_string(tokens)
    # # logging.info(f"토큰 > 최종적으로 사람이 읽을 수 있는 문자열로 복원 : {answer}")

    # 확률 분포 계산
    start_probs = torch.nn.functional.softmax(outputs.start_logits, dim=-1)
    end_probs = torch.nn.functional.softmax(outputs.end_logits, dim=-1)

    # 상위 후보 추출
    start_top = torch.topk(start_probs, k=top_k)
    end_top = torch.topk(end_probs, k=top_k)

    answers = []
    for i in range(top_k):
        for j in range(top_k):
            start_idx = start_top.indices[0][i].item()
            end_idx = end_top.indices[0][j].item() + 1
            if end_idx <= start_idx or end_idx - start_idx > max_answer_len:
                continue
            answer = qa_tokenizer.decode(
                inputs["input_ids"][0][start_idx:end_idx],
                skip_special_tokens=True
            )
            score = start_top.values[0][i].item() * end_top.values[0][j].item()
            answers.append((answer, score))

    # 확률 기준으로 정렬
    answers = sorted(answers, key=lambda x: x[1], reverse=True)

    return answers

# 형태소 분석
mecab = MeCab()
# 후처리 함수
def clean_result(text: str):
    text = text.strip() # 공백 제거
    text = text.replace("[CLS]", "").replace("[SEP]", "").replace("[PAD]", "") # 특수문자 토큰 제거

    morphs = mecab.pos(text)
    # 변형할 품사가 없으면 원본 반환
    if not morphs:
        return text
    
    # 마지막 단어/품사
    last_word, last_pos = morphs[-1]

    # 동사/형용사 처리
    if last_pos.startswith("VV") or last_pos.startswith("VA") or "XSV" in last_pos:
        if not text.endswith("다."):
            if text.endswith("다"):
                text += "."
            else:
                text += "다."
    
    # 명사 처리
    elif last_pos.startswith("NN"):
        if not text.endswith("이다."):
            text += "이다."
    
    # 조사로 끝나는 경우 -> 직전 형태소 확인
    elif last_pos.startswith("J"):
        prev_word, prev_pos = morphs[-2]
        if prev_pos.startswith("NN"): # 명사
            text += "이다."
        elif prev_pos.startswith("VV") or prev_pos.startswith("VA"): # 동사/형용사
            text += "다."
    
    # 기본 처리
    else:
        if not text.endswith("다."):
            if text.endswith("다"):
                text += "."
            else:
                text += "다."
    
    return text

# 후처리 로직: 불필요한 문자열 제거
def clean_qa(answers: list[str]):
    cleaned = []
    for ans in answers:
        text = ans.strip()
        # 출처만 남은 경우 제거
        if re.match(r'^\d+\s*(연합뉴스|네이트|KBS|매일경제|한겨레|조선일보)$', text):
            continue
        # 질문 반복 제거
        if "질문" in text or "알려줘" in text:
            continue
        cleaned.append(text)
    
    # 번호 제거 + 불필요한 공백 정리
    sentences = [re.sub(r'^\d+\s*', '', ans).strip() for ans in cleaned]
    sentences = [re.sub(r'\s+', ' ', s) for s in sentences]  # 띄어쓰기 교정
    
    # 문장 연결
    qa = " ".join(sentences)

    # 불필요한 도메인/기호 제거
    qa = re.sub(r'v\. ?da.*', '', qa)
    qa = re.sub(r'(https?://)?\S+\.(com|net|co\.kr)', '', qa).strip()
    qa = qa.replace("…", "...")

    return qa


# qdrant 검색 결과
query = "이란 전쟁 상황에 대해 알려줘"
search_result = search_qdrant(query)
for r in search_result:
    logging.info(
        f"Score: {r.score:.4f}, "
        f"Title: {r.payload.get('title')}, "
        f"Source: {r.payload.get('source_name')}, "
        f"URL: {r.payload.get('url')}, "
        f"Content: {r.payload.get('content')}"
    )

# 검색된 문서들을 context로 합치기
contexts = [r.payload.get('content') for r in search_result]
# 여러 문서들을 하나의 문자열로 합친다
context = " ".join(contexts)
logging.info(f"context : {context}")

# QA 모델: 질문(question)/문서(context)를 입력으로 받아서 답변을 복원 또는 생성
qa_answers = []
for i, doc in enumerate(contexts):
    # qa 모델 파이프라인(질문, 문서)
    ans = qa_pipeline(question=query, context=doc, top_k=3)

    if ans and ans not in ["[CLS]", ""]:
        # logging.info(f"clean_result without : {ans}")
        # qa_answers.append(f"{i+1} {clean_result(ans)}")
        qa_answers.append(f"{i+1} {ans}")
logging.info(f"qa_answers 형태소 분석 포함 후처리 : {qa_answers}")

# 최종 답변 선택
if qa_answers:
    final_qa_answers = qa_answers
else:
    # QA 모델 답변 실패시 검색 문서 전체를 fallback으로 사용
    final_qa_answers = context
logging.info(f"질문 : {query}")
logging.info(f"답변 : {final_qa_answers}")

# 메모리 정리: 객체 삭제, 메모리 반환
del qa_tokenizer
del qa_model
del embedder_model

gc.collect()
# GPU cuda 사용시 메모리 반환
if device=="cuda":
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

2026-05-19 14:34:34,989 [INFO] PyTorch Version : 2.2.2, Device : cpu
2026-05-19 14:34:35,009 [INFO] Use pytorch device_name: cpu


2026-05-19 14:34:35,010 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-19 14:34:43,407 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/query "HTTP/1.1 200 OK"
2026-05-19 14:34:43,411 [INFO] Score: 0.7292, Title: “미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯” - KBS 뉴스, Source: KBS 뉴스, URL: https://news.google.com/rss/articles/CBMiW0FVX3lxTE1HQ0U4UllMYVlWMkRqWGJrNVBieWVqWkFhS01fR3BpVFE3TTZLcWx2bU82cWZGczRYQ0V4LWwwbVFBVkNrRGQwQ3h3bVA1dGxYVm43ck04YmFVbkE?oc=5, Content: “미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯”  KBS 뉴스사우디, 이란 본토에 비밀 보복 공습…중동 긴장 고조  v.daum.net"미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯"  연합뉴스사우디, 중동 전쟁 중 이란에 공격…직접 군사 행동은 처음  newsis.com사우디, 전쟁 중 이란 직접 타격…이후 긴장 완화 합의  한겨레
2026-05-19 14:34:43,413 [INFO] Score: 0.6809, Title: 종전안 ‘쓰레기’라는 트럼프…부통령은 “이란과 진전 있다” - 매일경제, Source: None, URL: https://news.google.com/rss/articles/CBMiUkFVX3lxTE81VVlTTlFIaUZzVXVRdTJYcDZ2THk5Um9QbFRiNFB1Rkc3NUI4dVFsOTVYSXpQcXg1cGxOMUs1Tk5yLVJobkpKZDBRUEdvcGE2Vmc?oc=5, Content: 종전안 ‘쓰레기’라는 트럼프…부통령은 “이란과 진전 있다”  매일경제오바마 "우리는 한 발의 미사일 발사하지 않고 이란 문제 해결"  newsis.com밴스 “핵 

AttributeError: 'tuple' object has no attribute 'strip'

In [39]:
# 요약 모델 적용 : embedding + qdrant + Summary Model
import torch
from sentence_transformers import SentenceTransformer
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration
from qdrant_client import QdrantClient
import logging
import re

# 디바이스 설정
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
logging.info(f"PyTorch Version : {torch.__version__}, Device : {device}")

# embedding(sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2)
# 기반 구조 : Microsoft의 MiniLM을 기반으로 한 경량 Transformer 모델
# 임베딩 크기 : 384차원 벡터로 문장르 표현 -> 빠르고 메모리 효율적
# 환용 분야 : 문장 유사도 계산, 의미 기반 검색(Semantic Search), RAG에서 질의와 문서 매칭, 클러스터링, 추천 시스템
embedder_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder_model = SentenceTransformer(embedder_name)

# 모델 로드(gogamza/kobart-summarization)
# Facebook BART를 한국어에 맞게 변형한 KoBart 기반 모델
summarizer_name = "gogamza/kobart-summarization"
summarizer_tokenizer = PreTrainedTokenizerFast.from_pretrained(summarizer_name)
summarizer_model = BartForConditionalGeneration.from_pretrained(summarizer_name)

# Qdrant 서버 연결
qdrant = QdrantClient(host="localhost", port=6333)


# embedding 함수
def generate_embedding(text: str):
    embedding = embedder_model.encode(text).tolist() # 임베딩 벡터 변환
    return embedding

# qdrant 검색 함수
def search_qdrant(query: str, top_k: int = 5):
    query_vector = generate_embedding(query) # 질의 임베딩 생성

    # Qdrant 검색
    qdrant_result = qdrant.query_points(
        collection_name="news_articles",
        query=query_vector,
        limit=top_k
    )
    # query_result : QueryResponse 객체, 검색 결과를 감싸는 객체
    # query_result.points : ScoredPoint 리스트 실제 결과는 query_result.points에 들어 있는 ScoredPoint 리스트, score, .payload 등을 꺼내 쓰면 된다
    return qdrant_result.points

# 요약 함수
def summarize_text(context, max_length=50, min_length=10):
    # 요약 토크나이저
    inputs = summarizer_tokenizer(
        [context],
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    # logging.info(f"요약 토크나이저 결과: {inputs}")
    # logging.info(f"max_length, min_length: {max_length}, {min_length}")


    # 요약 생성
    summary_ids = summarizer_model.generate(
        # 요약 기본 옵션
        inputs["input_ids"], # 토크나이저가 변환한 입력 문장 토큰, 모델이 요약을 생성할때 조건으로 사용
        num_beams=4, # Beam Search 탐색 개수(값이 클수록 다양한 후보를 탐색해 더 좋은 요약을 얻을수 있으나 속도가 느려진다)
        max_length=max_length, # 생성할 요약의 최대 토큰 길이
        min_length=min_length, # 생성할 요약의 최소 토큰 길이
        early_stopping=True, # 모든 Beam이 EOS(End of Sentence) 토큰에 도달하면 탐색을 조기 종료

        # 반복 결과 제한 옵션
        no_repeat_ngram_size=3, # 특정 길이의 n-gram(연속된 단어 묶음)이 반복되지 않도록 제한
        repetition_penalty=1.5, # 동일 단어를 반복할 경우 확률을 낮추는 패널티 적용(1.2~2.0)
        length_penalty=2.0, # 너무 짧거나 긴 문장을 방지하기 위해 길이에 따른 점수 조정(>1.0 더 긴문장 선호, <1.0 짧은 문장 선호)

        # 창의적, 다양성 옵션
        do_sample=True, # 확률 분포에서 샘플링(결과가 매번 달라 질 수 있고, 창의적/다양한 요약 생성 가능)
        temperature=0.9, # 확률 분포 조정, 1.0(원래 확률 분포 그대로)
        top_p=0.85 # 누적 확률이 85%에 해당하는 상위 토큰 집합에서만 샘플링
    )

    # 요약 Decode 한글로 변환
    summary = summarizer_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

    return summary

# 후처리 로직: 불필요한 문자열 제거
def clean_summary(summary: str):
    summary = re.sub(r'(https?://)?\S+\.(com|net|co\.kr)', '', summary).strip()
    return summary


# Qdrant 검색
query = "이란 전쟁 상황에 대해 알려줘"
search_result = search_qdrant(query=query)
for r in search_result:
    logging.info(
        f"Score: {r.score:.4f}, "
        f"Title: {r.payload.get('title')}, "
        f"Source: {r.payload.get('source_name')}, "
        f"URL: {r.payload.get('url')}, "
        f"Content: {r.payload.get('content')}"
    )

# 검색된 문서들을 context로 합치기
contexts = [r.payload.get('content') for r in search_result]
# 여러 문서들을 하나의 문자열로 합치기
context = " ".join(contexts)
logging.info(f"여러 문서들을 하나의 문자열로: {context}")

# 요약: 검색문서들을 요약
summary = summarize_text(context=context, max_length=100, min_length=40)
# logging.info(f"검색문서들을 요약: {summary}")

# 요약: 요약 문서들을 후처리
summary = clean_summary(summary=summary)
logging.info(f"요약 문서들을 후처리: {summary}")

# 요약: 검색 질의문 + 검색 결과
combined_input = f"질문: {query}\n답변: {context}"
qdrant_summary = summarize_text(context=combined_input, max_length=100, min_length=40)
qdrant_summary = clean_summary(summary=qdrant_summary)
logging.info(f"검색 질의문 + 검색 결과들을 요약: {qdrant_summary}")

2026-05-19 13:29:27,937 [INFO] PyTorch Version : 2.2.2, Device : cpu
2026-05-19 13:29:28,146 [INFO] Use pytorch device_name: cpu
2026-05-19 13:29:28,148 [INFO] Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BartTokenizer'. 
The class this function is called from is 'PreTrainedTokenizerFast'.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
2026-05-19 13:29:38,011 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-05-19 13:29:38,203 [INFO] HTTP Request: POST http://localhost:6333/collections/news_articles/points/query "HTTP/1.1 200 OK"
2026-05-19 13:29:38,210 [INFO] Score: 0.7292, Title: “미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯” - KBS 뉴스, Source: KBS 뉴스, URL: https://news.google.com/rss/articles/CBMiW0FVX3lxTE1HQ0U4UllMYVlWMkRqWGJrNVBieWVqWkFhS01fR3BpVFE3TTZLcWx2bU82cWZGczRYQ0V4LWwwbVFBVkNrRGQwQ3h3bVA1dGxYVm43ck04YmFVbkE?oc=5, Content: “미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯”  KBS 뉴스사우디, 이란 본토에 비밀 보복 공습…중동 긴장 고조  v.daum.net"미국·이란 휴전 깨지면 UAE 가장 먼저 전쟁 휘말릴 듯"  연합뉴스사우디, 중동 전쟁 중 이란에 공격…직접 군사 행동은 처음  newsis.com사우디, 전쟁 중 이란 직접 타격…이후 긴장 완화 합의  한겨레
2026-05-19 13:29:38,212 [INFO] Score: 0.6809, Title: 종전안 ‘쓰레기’라는 트럼프…부통령은 “이란과 진전 있다” - 매일경제, Source: None, URL: https://news.google.com/rss/articles/CBMiUkFVX3lxTE81VVlTTlFIaUZzVXVRdTJYcDZ2THk5Um9QbFRiNFB1Rkc3NUI4dVFsOTVYSXpQcXg1cGxOMUs1Tk5yLVJobkpKZDBRUEdvcGE2Vmc?oc=5, Content: 종전안 ‘쓰레기’라는 트럼프…부통령은 “이란과 진전 있다”  매일경제오바마 "우리는 한 발의 미사일 발사하지 않고 이란 문제 해결"  newsis.com밴스 “핵 